In [5]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn import functional as F

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 基本配置
BASE_DIR = Path('/home/thelia/Learn/ML/MyCode/LearningMachineLearning/Thelia/ProjectSekai')
TRAINING_DATA_DIR = BASE_DIR / 'TrainingData'
AUGMENTED_DATA_DIR = BASE_DIR / 'AugmentedTrainingData'
TRAIN_DATA_DIR = AUGMENTED_DATA_DIR / 'train'
TEST_DATA_DIR = AUGMENTED_DATA_DIR / 'test'
CODE_DIR = BASE_DIR / 'Codes'

# 超参数
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5
RANDOM_SEED = 42
TEST_SPLIT_RATIO = 0.2

# 类别定义
CLASS_NAMES = ['Enanan', 'Kanade', 'Mafuyu', 'Mizuki']
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

print(f"配置信息:")
print(f"  原始训练数据目录: {TRAINING_DATA_DIR}")
print(f"  增强后数据目录: {AUGMENTED_DATA_DIR}")
print(f"  训练集目录: {TRAIN_DATA_DIR}")
print(f"  测试集目录: {TEST_DATA_DIR}")
print(f"  类别数: {NUM_CLASSES}")
print(f"  类别: {CLASS_NAMES}")
print(f"  批大小: {BATCH_SIZE}")
print(f"  学习率: {LEARNING_RATE}")
print(f"  最大epoch数: {NUM_EPOCHS}")
print(f"  测试集比例: {TEST_SPLIT_RATIO}")

使用设备: cpu
配置信息:
  原始训练数据目录: /home/thelia/Learn/ML/MyCode/LearningMachineLearning/Thelia/ProjectSekai/TrainingData
  增强后数据目录: /home/thelia/Learn/ML/MyCode/LearningMachineLearning/Thelia/ProjectSekai/AugmentedTrainingData
  训练集目录: /home/thelia/Learn/ML/MyCode/LearningMachineLearning/Thelia/ProjectSekai/AugmentedTrainingData/train
  测试集目录: /home/thelia/Learn/ML/MyCode/LearningMachineLearning/Thelia/ProjectSekai/AugmentedTrainingData/test
  类别数: 4
  类别: ['Enanan', 'Kanade', 'Mafuyu', 'Mizuki']
  批大小: 32
  学习率: 0.001
  最大epoch数: 15
  测试集比例: 0.2


In [ ]:
# ==================== Phase 0: 数据增强 + 8:2 切分 ====================
# 规则：先按原始图片分 train/test，再对每张原图生成 6 个变体，避免数据泄露

import random
import shutil

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def make_six_variants(img):
    """对一张图片生成 6 个变体。"""
    return [
        ("original", img),
        ("h_flip", img.transpose(Image.FLIP_LEFT_RIGHT)),
        ("v_flip", img.transpose(Image.FLIP_TOP_BOTTOM)),
        ("hv_flip", img.transpose(Image.FLIP_LEFT_RIGHT).transpose(Image.FLIP_TOP_BOTTOM)),
        ("rot90", img.rotate(90, expand=True)),
        ("rot270", img.rotate(270, expand=True)),
    ]


def prepare_output_dirs(base_dir, class_names):
    """创建 train/test 输出目录。"""
    if base_dir.exists():
        shutil.rmtree(base_dir)
    for split_name in ['train', 'test']:
        for class_name in class_names:
            (base_dir / split_name / class_name).mkdir(parents=True, exist_ok=True)


def split_files(file_list, test_ratio=0.2, seed=42):
    """按文件级别切分 train/test，返回 (train_files, test_files)。"""
    file_list = list(file_list)
    rng = random.Random(seed)
    rng.shuffle(file_list)
    test_size = max(1, int(round(len(file_list) * test_ratio))) if len(file_list) > 1 else len(file_list)
    test_files = file_list[:test_size]
    train_files = file_list[test_size:]
    return train_files, test_files


def get_source_files(class_source_dir):
    """获取原始 JPEG 文件，兼容 .jpg 和 .jpeg。"""
    jpg_files = sorted(class_source_dir.glob('*.jpg'))
    jpeg_files = sorted(class_source_dir.glob('*.jpeg'))
    return sorted(jpg_files + jpeg_files)


def augment_and_save_split(source_files, output_dir, class_name, split_name):
    """对给定 split 的原图生成 6 个 JPEG 变体。"""
    target_dir = output_dir / split_name / class_name
    target_dir.mkdir(parents=True, exist_ok=True)
    generated = 0
    for source_file in tqdm(source_files, desc=f"{class_name}-{split_name}"):
        try:
            img = Image.open(source_file).convert('RGB')
            base_name = source_file.stem
            for variant_name, variant_img in make_six_variants(img):
                variant_img.save(
                    target_dir / f'{base_name}_{variant_name}.jpg',
                    'JPEG',
                    quality=95,
                    optimize=True,
                )
                generated += 1
        except Exception as exc:
            print(f"  错误处理 {source_file.name}: {exc}")
    return generated


print("=" * 60)
print("开始数据增强：单独输出目录 + 原图级别8:2切分 + 每图6个变体")
print("=" * 60)

prepare_output_dirs(AUGMENTED_DATA_DIR, CLASS_NAMES)

total_train_generated = 0
total_test_generated = 0
source_summary = {}

for class_name in CLASS_NAMES:
    class_source_dir = TRAINING_DATA_DIR / class_name
    source_files = get_source_files(class_source_dir)
    source_summary[class_name] = len(source_files)

    if not source_files:
        print(f"⚠ {class_name} 目录中没有找到 .jpg 或 .jpeg 文件: {class_source_dir}")
        continue

    train_files, test_files = split_files(source_files, test_ratio=TEST_SPLIT_RATIO, seed=RANDOM_SEED)
    print(f"\n{class_name}: 原始图片 {len(source_files)} 张, train {len(train_files)} 张, test {len(test_files)} 张")

    train_count = augment_and_save_split(train_files, AUGMENTED_DATA_DIR, class_name, 'train')
    test_count = augment_and_save_split(test_files, AUGMENTED_DATA_DIR, class_name, 'test')

    total_train_generated += train_count
    total_test_generated += test_count

    print(f"✓ {class_name}: 训练集生成 {train_count} 张，测试集生成 {test_count} 张")

print("\n" + "=" * 60)
print("数据增强完成！")
print(f"训练集总计: {total_train_generated} 张JPEG图片")
print(f"测试集总计: {total_test_generated} 张JPEG图片")
print("=" * 60)

print("\n数据集统计:")
for split_name in ['train', 'test']:
    for class_name in CLASS_NAMES:
        class_dir = AUGMENTED_DATA_DIR / split_name / class_name
        jpg_count = len(list(class_dir.glob('*.jpg'))) if class_dir.exists() else 0
        print(f"  {split_name}/{class_name}: {jpg_count} 张JPEG图片")


开始数据增强：单独输出目录 + 原图级别8:2切分 + 每图6个变体

Enanan: 原始图片 87 张, train 70 张, test 17 张


Enanan-test: 100%|██████████| 17/17 [00:00<00:00, 21.39it/s]


✓ Enanan: 训练集生成 420 张，测试集生成 102 张

Kanade: 原始图片 87 张, train 70 张, test 17 张


Kanade-train:  43%|████▎     | 30/70 [00:01<00:01, 20.76it/s]

In [ ]:
# 调试：检查数据目录结构
print("调试：检查目录结构")
print(f"TRAINING_DATA_DIR: {TRAINING_DATA_DIR}")
print(f"存在: {TRAINING_DATA_DIR.exists()}")

if TRAINING_DATA_DIR.exists():
    print(f"\nTRAINING_DATA_DIR 内容:")
    for item in TRAINING_DATA_DIR.iterdir():
        print(f"  {item.name} ({'目录' if item.is_dir() else '文件'})")
        if item.is_dir():
            contents = list(item.glob('*'))
            print(f"    包含 {len(contents)} 项")
            if len(contents) > 0:
                print(f"    样例: {contents[0].name}")
else:
    print("TRAINING_DATA_DIR 不存在！")
